# DemandIQ: Data Reconnaissance

This notebook establishes what the Rossmann data contains and which constraints matter before modeling. It is intentionally descriptive: it does not perform exploratory analysis, feature engineering, imputation, outlier removal, modeling, or dataset creation.

Raw CSV files are loaded read-only. Any date conversion is kept in analysis copies.

In [ ]:
from pathlib import Path

import pandas as pd

# Locate the repository without depending on the notebook execution directory.
project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'data' / 'raw' / 'rossmann').is_dir()
)
data_dir = project_root / 'data' / 'raw' / 'rossmann'

train = pd.read_csv(data_dir / 'train.csv', low_memory=False)
test = pd.read_csv(data_dir / 'test.csv', low_memory=False)
store = pd.read_csv(data_dir / 'store.csv', low_memory=False)
sample_submission = pd.read_csv(data_dir / 'sample_submission.csv', low_memory=False)

print(f'Project root: {project_root}')
print(f'Loaded train={train.shape}, test={test.shape}, store={store.shape}, sample_submission={sample_submission.shape}')

## 1. Dataset Overview

The three modeling tables are summarized together so each table's structure can be compared without repeating separate inspections.

In [ ]:
tables = {'train': train, 'test': test, 'store': store}
overview = pd.DataFrame(
    {
        name: {
            'rows': frame.shape[0],
            'columns': frame.shape[1],
            'column_names': list(frame.columns),
            'memory_mb': round(frame.memory_usage(deep=True).sum() / 1024**2, 2),
        }
        for name, frame in tables.items()
    }
).T
display(overview)

print('Data types')
display(pd.concat({name: frame.dtypes.astype(str) for name, frame in tables.items()}, axis=1))

for name, frame in tables.items():
    print(f'First five rows: {name}')
    display(frame.head())

In [ ]:
print('Submission template schema')
display(sample_submission.head())
print('Columns:', list(sample_submission.columns))

## 2. Target and Feature Structure

`Sales` is the prediction target because it is the numeric outcome present in the training labels and the submission template asks for one `Sales` value for each test `Id`. It is absent from `test`, so it is not known at prediction time.

`Customers` is historical training information. It is present in `train` but absent from `test`, so it must not be used as a future prediction feature.

In [ ]:
train_columns = set(train.columns)
test_columns = set(test.columns)
print('Train columns:', list(train.columns))
print('Test columns:', list(test.columns))
print('Only in train:', sorted(train_columns - test_columns))
print('Only in test:', sorted(test_columns - train_columns))
print('Shared:', sorted(train_columns & test_columns))
print('Store-level attributes:', list(store.columns))
print('Sample submission columns:', list(sample_submission.columns))

assert 'Sales' in train_columns and 'Sales' not in test_columns
assert 'Customers' in train_columns and 'Customers' not in test_columns

## 3. Date Structure

The source `Date` columns are text. For analysis, parsed copies are created below; the loaded raw frames and CSV files are not overwritten.

In [ ]:
train_analysis = train.assign(Date=pd.to_datetime(train['Date']))
test_analysis = test.assign(Date=pd.to_datetime(test['Date']))
date_summary = pd.DataFrame(
    {
        'train': {
            'source_type': str(train['Date'].dtype),
            'analysis_type': str(train_analysis['Date'].dtype),
            'minimum_date': train_analysis['Date'].min(),
            'maximum_date': train_analysis['Date'].max(),
            'unique_dates': train_analysis['Date'].nunique(),
        },
        'test': {
            'source_type': str(test['Date'].dtype),
            'analysis_type': str(test_analysis['Date'].dtype),
            'minimum_date': test_analysis['Date'].min(),
            'maximum_date': test_analysis['Date'].max(),
            'unique_dates': test_analysis['Date'].nunique(),
        },
    }
).T
display(date_summary)
chronologically_separated = train_analysis['Date'].max() < test_analysis['Date'].min()
print('Train and test are chronologically separated:', chronologically_separated)
assert chronologically_separated

## 4. Store Structure

Store IDs are checked against the store dimension table. This establishes join coverage without analyzing `StoreType` or `Assortment` in depth.

In [ ]:
train_stores = set(train['Store'])
test_stores = set(test['Store'])
store_ids = set(store['Store'])
store_structure = pd.Series(
    {
        'unique stores in train': train['Store'].nunique(),
        'unique stores in test': test['Store'].nunique(),
        'unique stores in store table': store['Store'].nunique(),
        'train IDs overlapping store table': len(train_stores & store_ids),
        'test IDs overlapping store table': len(test_stores & store_ids),
        'train IDs without store record': len(train_stores - store_ids),
        'test IDs without store record': len(test_stores - store_ids),
    },
)
display(store_structure.to_frame('value'))
assert not (train_stores - store_ids) and not (test_stores - store_ids)

## 5. Missing Values

Missingness is reported only. No values are imputed and no rows or columns are deleted.

In [ ]:
def missing_report(frame: pd.DataFrame) -> pd.DataFrame:
    missing_count = frame.isna().sum()
    return pd.DataFrame(
        {'missing_count': missing_count, 'missing_percentage': (missing_count / len(frame) * 100).round(3)}
    ).query('missing_count > 0')

for name, frame in tables.items():
    print(f'Missing values: {name}')
    report = missing_report(frame)
    display(report if not report.empty else pd.DataFrame({'missing_count': [0], 'missing_percentage': [0.0]}, index=['none']))

## 6. Duplicates and Structural Integrity

Duplicate records are measured but not removed. Store-Date is the expected observation key for the daily sales tables; test `Id` is checked as its row identifier.

In [ ]:
integrity_report = pd.DataFrame(
    {
        'train': {
            'duplicate_full_rows': train.duplicated().sum(),
            'duplicate_store_date': train.duplicated(['Store', 'Date']).sum(),
            'unique_store_date_pairs': train[['Store', 'Date']].drop_duplicates().shape[0],
        },
        'test': {
            'duplicate_full_rows': test.duplicated().sum(),
            'duplicate_store_date': test.duplicated(['Store', 'Date']).sum(),
            'duplicate_ids': test['Id'].duplicated().sum(),
            'unique_store_date_pairs': test[['Store', 'Date']].drop_duplicates().shape[0],
        },
        'store': {
            'duplicate_full_rows': store.duplicated().sum(),
            'duplicate_store_ids': store['Store'].duplicated().sum(),
        },
    }
).T
display(integrity_report)
print('Observation: zero duplicate counts indicate no duplicates were detected in these checks; this notebook does not remove any records.')

## 7. Train/Test Relationship

Training contains historical outcomes (`Sales`) and the historical related variable (`Customers`), plus date, store, promotion, holiday, and opening-status fields. Prediction contains the test-time fields and the store dimension attributes can be joined by `Store`; it does not contain future `Sales` or `Customers`.

The main leakage risks are using `Sales` or `Customers` from the forecast period, or using any value that would only be observed after the prediction date. `Date`, `Store`, and scheduled promotion/holiday fields can be candidates only when their timing and availability are respected. Historical information means data observed before the forecast origin; future information means an outcome or measurement unavailable at that origin. This distinction must guide later validation and feature preparation.

In [ ]:
availability = pd.DataFrame(
    {
        'Sales': {'training': 'available', 'prediction': 'unavailable', 'risk': 'target / future outcome'},
        'Customers': {'training': 'available', 'prediction': 'unavailable', 'risk': 'historical variable; future leakage if used directly'},
        'Date': {'training': 'available', 'prediction': 'available', 'risk': 'must respect forecast timing'},
        'Store': {'training': 'available', 'prediction': 'available', 'risk': 'join key; must have dimension coverage'},
        'Promo and holiday fields': {'training': 'available', 'prediction': 'available', 'risk': 'availability depends on forecast scenario'},
    }
).T
display(availability)

## 8. Initial Data Quality Assessment

**Observed:** `Date` is initially text; `test.Open` has missing values; the store table has missing competition and promotion timing attributes; and the target and customer history are train-only. The checked Store IDs have complete store-table coverage, and no full-row or checked identifier duplicates were detected.

**Worth investigating later:** the meaning and handling of missing operational attributes, the missing `Open` values, and unusual values in numeric and categorical fields. These are reconnaissance observations, not fixes.

**Limitations:** this notebook does not assess distributions, relationships, causality, outliers, or model performance. Those belong to later EDA and data-preparation work.

In [ ]:
quality_flags = pd.Series(
    {
        'date_requires_parsing_for_time_analysis': train['Date'].dtype == 'object' and test['Date'].dtype == 'object',
        'test_has_missing_values': test.isna().any().any(),
        'store_has_missing_values': store.isna().any().any(),
        'train_store_coverage_complete': not (train_stores - store_ids),
        'test_store_coverage_complete': not (test_stores - store_ids),
        'train_test_chronologically_separated': chronologically_separated,
    },
)
display(quality_flags.to_frame('observed'))

## 9. Key Findings and Implications

### 1. What we know
- `train` contains daily store observations from 2013-01-01 through 2015-07-31, including `Sales` and `Customers`.
- `test` covers 2015-08-01 through 2015-09-17 and excludes both outcomes.
- Store records cover every store appearing in train and test.

### 2. What needs investigation in EDA
- The meaning and distribution of missing `Open` and store competition/promotion attributes.
- Suspicious or unusual numeric, categorical, holiday, and promotion values.
- Temporal and store-level patterns, without treating association as causation.

### 3. What must be handled during data preparation
- Parse dates in analysis or modeling copies and define a time-aware validation strategy.
- Decide how missing values will be handled using rules available at prediction time.
- Join store attributes by `Store` while preserving row coverage and identifiers.

### 4. What could create data leakage
- Using `Sales` or `Customers` from the forecast period, or any post-forecast observation.
- Random validation that allows future observations to inform past predictions.
- Treating fields as known merely because they exist in a historical row; availability at forecast time must be established.

### 5. Important modeling constraints
- The prediction target is `Sales`, and `Customers` must not be used as a future prediction feature.
- The train/test periods are chronologically separated, so temporal ordering is a core constraint.
- Missingness, scheduled-versus-observed information, and the test submission identifier must be addressed during later preparation.

No EDA, feature engineering, outlier removal, imputation, model training, or processed dataset creation was performed here.